<!-- dads-lab-header -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mdehghani86/DADS5250-GenAI/blob/main/labs/M12/M12_Lab1_OpenAI_Agents_SDK.ipynb)

![Module 12 Lab 1 - OpenAI Agents SDK](https://raw.githubusercontent.com/mdehghani86/DADS5250-GenAI/main/labs/M12/assets/images/M12_Lab1_OpenAI_Agents_SDK_banner.png)

In [ ]:
# === Shared lab setup: install dads5250 + the OpenAI Agents SDK, then import ===
# Installs the shared utilities (pp, pretty_print, lab_pill, model constants,
# setup_openai) AND the `openai-agents` package once per Colab runtime. The same
# OPENAI_API_KEY Colab secret is used across every DADS 5250 lab. Set it once in
# the key sidebar and it is picked up automatically.
import os
import importlib.util
# One combined install: the shared utils package + the Agents SDK it drives.
!pip install -q dads5250==0.2.0 openai-agents

from dads5250 import (
    pp,
    pretty_print,
    lab_pill,
    setup_openai,
    DEFAULT_CHAT_MODEL,
    DEFAULT_MINI_MODEL,
)

lab_pill('M12 Lab 1: OpenAI Agents SDK')   # sticky banner so you always see which lab you're in

## API check

Confirm the API connection before we start. `setup_openai()` loads your key and, importantly, also writes it into `os.environ["OPENAI_API_KEY"]`. The Agents SDK reads that environment variable directly, so once this cell runs green, every `Agent` and `Runner` below can talk to the model with no extra configuration. Your key comes from a Colab Secret, an environment variable, or a hidden prompt if neither is set.

**Running in Jupyter or JupyterHub instead of Colab?** There are no Colab Secrets there, so do either:
- Just run the setup cell and paste your key when it prompts (the input is hidden and not saved), or
- Set it first: in a terminal run `export OPENAI_API_KEY=sk-...`, or in a cell run `import os; os.environ["OPENAI_API_KEY"] = "sk-..."`.


In [ ]:
# === API check: confirm the connection and expose the key to the Agents SDK ===
client = setup_openai()                       # loads + verifies OPENAI_API_KEY
os.environ["OPENAI_API_KEY"] = client.api_key # the Agents SDK reads the key from the environment

pp({
    "OpenAI":       "connected",
    "agents SDK":   "reads OPENAI_API_KEY from os.environ",
    "agent model":  DEFAULT_MINI_MODEL,        # the model our agents will run on
}, title="API check")

# 🤖 From Tools to Teams: the OpenAI Agents SDK

**Why this matters.** In Module 3 you wired a model to a single tool by hand: you wrote the JSON schema, sent the `tools` list, caught the tool call, ran the function, and fed the result back in a loop. That taught you *exactly* what is happening under the hood, and it works. But writing that call-execute-return loop yourself for every project, then bolting on retries, routing between agents, and safety checks, quickly becomes a lot of plumbing.

**The Agents SDK is that plumbing, done for you.** It is a small, opinionated Python framework (the production successor to OpenAI's experimental "Swarm") that turns the loop you hand-wrote into a handful of primitives. You describe *what* your system is — an **Agent** with instructions, some **Tools**, maybe some **Handoffs** to specialists, and **Guardrails** for safety — and a **Runner** executes the whole agentic loop for you and hands back the final answer.

**Minimalism is the point.** The SDK deliberately has a tiny surface area: a few powerful pieces you compose with plain Python. If you know Python functions and classes, you already know most of it. In this lab we climb the ladder one rung at a time: your first agent, then tools, then a multi-agent handoff, then a brief look at guardrails. By the end you can stand up a small multi-agent app in a few lines.

## 🧩 1. What the Agents SDK is (and why, vs the raw API)

Strip the SDK down and it is four nouns and one verb:

- **Agent** — a model configured with a `name`, `instructions` (its job description), and optionally `tools` and `handoffs`.
- **Tools** — plain Python functions the agent may call, marked with a `@function_tool` decorator so the SDK can read their name, arguments, and docstring automatically.
- **Handoffs** — other agents this agent is allowed to delegate to, so one coordinator can route work to specialists.
- **Guardrails** — input/output checks that run alongside the agent and fail fast on unsafe or off-topic requests.
- **Runner** (the verb) — the execution engine. It takes an agent plus an input, runs the tool-calling and handoff loop until the task is done, and returns a result.

**Why not just use the raw Chat Completions API (like M3)?** You *can*, and you did. The difference is who writes the loop. With the raw API, **you** manage the message list, detect tool calls, dispatch functions, append results, and re-call the model — for every tool and every turn. The SDK folds all of that into `await Runner.run(...)`. It also builds the tool JSON schema *from your function's type hints and docstring*, so you never hand-write a schema again. Same mechanics you already understand — far less boilerplate.

## 🚀 2. Your first agent (Agent + Runner)

The smallest useful program is an **Agent** and a **Runner**. The Agent is a description; the Runner makes it go.

You give the agent a `name` (for logs and handoffs), `instructions` (its system prompt, in plain English), and a `model`. Then you run it with **`await Runner.run(agent, "...")`**, which drives the agentic loop and returns a `result` object; the answer lives in `result.final_output`. No tools yet, just prove the wiring works end to end.

> **Why `await Runner.run(...)` and not `Runner.run_sync(...)`?** Colab and Jupyter already run an asyncio event loop, and this SDK deliberately refuses to start a second one, so the synchronous `run_sync` raises `RuntimeError: ... event loop is already running` inside a notebook. Notebooks support top-level `await`, so we call the async method `Runner.run(...)` directly. In a plain `.py` script (no running loop) you would use `Runner.run_sync(...)` instead. Same loop, two entry points.

In [ ]:
# ==========================================================
# 2. Your first agent: Agent + Runner
# ==========================================================
from agents import Agent, Runner            # the two core primitives

# An Agent = model + a name + instructions (its system prompt in plain English)
assistant = Agent(
    name="Assistant",                                        # used in logs and handoffs
    instructions="You are a concise, friendly helper. Answer in one or two sentences.",
    model=DEFAULT_MINI_MODEL,                                # fast + cheap model for our agents
)

# await Runner.run drives the whole agentic loop and returns a result object
result = await Runner.run(assistant, "In one sentence, what is an AI agent?")

# result.final_output is the finished answer text
pretty_print(result.final_output, title="Your first agent's answer")

## 🔧 3. Tools with `@function_tool`

An agent that can only talk is a chatbot. An agent that can **act** needs tools. In M3 you hand-wrote a JSON schema for every function. The SDK removes that chore: you decorate an ordinary Python function with **`@function_tool`**, and the SDK reads the function's **name**, its **type-hinted arguments**, and its **docstring** to build the schema for you automatically.

Two rules make this work well:
1. **Add type hints** (`city: str`) — they become the argument schema the model fills in.
2. **Write a real docstring** — it becomes the tool's description, which is how the model decides *when* to call it. Treat it like documentation aimed at the model.

Below we give the agent a `get_weather` tool. When you ask about weather, the Runner will call the tool, get the value, and let the agent finish the answer with real data — the exact call-execute-return loop from M3, now handled for you.

In [ ]:
# ==========================================================
# 3. A tool: a plain function + the @function_tool decorator
# ----------------------------------------------------------
# Defines:
#   - get_weather() : Get the current weather for a given city.
# ==========================================================
from agents import function_tool            # the decorator that turns a function into a tool

@function_tool                              # <- this line registers the function as a tool
def get_weather(city: str) -> str:          # type hints become the argument schema
    """Get the current weather for a given city."""   # docstring becomes the tool description
    # In a real app this would call a weather API. We fake it so the lab always runs.
    fake = {                                # a tiny lookup so results are deterministic
        "Boston": "58°F and cloudy",
        "Miami": "84°F and sunny",
        "Seattle": "51°F and rainy",
    }
    return fake.get(city, f"70°F and clear in {city}")     # default for any other city

# Build an agent that is ALLOWED to use the tool (pass it in the tools list)
weather_agent = Agent(
    name="WeatherAssistant",
    instructions="You help with weather questions. Use the get_weather tool when asked about a city.",
    model=DEFAULT_MINI_MODEL,
    tools=[get_weather],                    # the agent may call this tool; the Runner handles the loop
)

# Ask a question that needs the tool. The Runner calls get_weather for us behind the scenes.
result = await Runner.run(weather_agent, "What's the weather in Boston?")
pretty_print(result.final_output, title="Tool-grounded answer")

## 🔎 3b. Watch the agentic loop the Runner ran

`Runner` felt like magic: you asked a question and got an answer. But under the hood it ran the exact call-execute-return loop you wrote by hand in M3. Every run records the steps it took in `result.new_items`, and reading them is the best way to *see* what an agent actually did: which tool it called, with what arguments, what came back, and the final message. Treat this as your debugger and your observability window.

In [ ]:
# ==========================================================
# 3b. Inspect result.new_items to SEE the tool call the SDK ran
# ----------------------------------------------------------
# Defines:
#   - trace_steps(result) : turn a run's new_items into readable step lines
# ==========================================================
from agents import ItemHelpers          # helper to pull text out of a message item

def trace_steps(result):
    """Turn a run's new_items into human-readable step lines."""
    steps = []
    for it in result.new_items:                  # each item = one thing the agent did
        t = getattr(it, "type", it.__class__.__name__)
        if t == "tool_call_item":                # the agent decided to CALL a tool
            raw = it.raw_item
            steps.append(f"tool call: {getattr(raw, 'name', '?')}({getattr(raw, 'arguments', '')})")
        elif t == "tool_call_output_item":        # the tool RETURNED a value
            steps.append(f"tool result: {getattr(it, 'output', '')}")
        elif t == "handoff_call_item":            # the agent asked to hand off
            steps.append("handoff requested")
        elif t == "handoff_output_item":          # control moved to another agent
            steps.append(f"handed off to: {getattr(getattr(it, 'target_agent', None), 'name', '?')}")
        elif t == "message_output_item":          # the agent produced final text
            steps.append(f"message: {ItemHelpers.text_message_output(it)[:90]}")
        else:
            steps.append(t)
    return steps

# 'result' is still the weather run from section 3; watch the loop it executed
pp(trace_steps(result), title="🔎 What the weather agent actually did")

> **Pause and think.** Notice what you did *not* write: no JSON schema, no `tool_calls` parsing, no `role: tool` message, no loop. The SDK read your function's signature and docstring and ran the whole call-execute-return cycle for you. Compare that to the M3 function-calling lab — same mechanics, a fraction of the code. Which tool in your own work would you register first, and what would its one-line docstring say so the model always picks it at the right moment?

**Your notes** *(double-click to edit)*

- A tool from my own work I would register first: 
- Its type-hinted arguments: 
- The one-line docstring that tells the model when to call it: 

## 🔀 4. Handoffs: a triage agent routing to specialists

One giant do-everything agent gets brittle fast. The SDK's signature idea is the **handoff**: build small **specialists**, then a **triage** agent that reads each request and *delegates* to the right one. Routing is decided at runtime, not hard-wired, so the same system handles many kinds of requests without you enumerating every path.

The mechanics are dead simple: an agent lists other agents in its `handoffs=[...]`, and its instructions tell it *when* to route to each. Below, a **Triage** agent routes billing questions to a **Billing** specialist and technical questions to a **Support** specialist. The whole thing is still **one run** — control moves between agents, but `await Runner.run` drives it start to finish.

In [ ]:
# ==========================================================
# 4. Handoffs: a triage agent routes to specialist agents
# ==========================================================
# Two small specialists, each with a narrow job described in its instructions
billing_agent = Agent(
    name="Billing",
    instructions="You handle billing questions: invoices, refunds, charges. Be precise and reassuring.",
    model=DEFAULT_MINI_MODEL,
)
support_agent = Agent(
    name="Support",
    instructions="You handle technical support: bugs, errors, how-to. Give clear step-by-step help.",
    model=DEFAULT_MINI_MODEL,
)

# The triage agent does NOT answer itself — it routes to the right specialist
triage_agent = Agent(
    name="Triage",
    instructions="Route the user to the right specialist. Billing questions -> Billing. "
                 "Technical or how-to questions -> Support.",
    model=DEFAULT_MINI_MODEL,
    handoffs=[billing_agent, support_agent],   # the agents Triage is allowed to delegate to
)

# One run, two different requests: watch which specialist ends up answering.
billing_result = await Runner.run(triage_agent, "I was charged twice for my subscription this month.")
support_result = await Runner.run(triage_agent, "The app crashes every time I click export. How do I fix it?")

pp({
    "billing question -> handled by": billing_result.last_agent.name,   # which agent produced the answer
    "billing answer":                 billing_result.final_output,
    "support question -> handled by": support_result.last_agent.name,
    "support answer":                 support_result.final_output,
}, title="Triage routed each request to the right specialist")

## 🛡️ 5. Guardrails (brief)

Real agents need boundaries. **Guardrails** are checks that run *alongside* an agent and **fail fast**:

- **Input guardrails** validate the request *before* the agent spends tokens on it — e.g. reject off-topic or unsafe input. They apply to the **first** agent that receives the request.
- **Output guardrails** validate the final answer *before* it reaches the user — e.g. block a response that leaks private data. They apply to the agent that produces the **final** output.

You attach them via `input_guardrails=[...]` and `output_guardrails=[...]` on an `Agent`. Each guardrail is a small function (decorated with `@input_guardrail` / `@output_guardrail`) that returns a `GuardrailFunctionOutput` saying whether the tripwire fired. When it fires, the SDK raises an exception and stops the run — so bad input never reaches the model and bad output never reaches the user. The cell below sketches the shape so you recognize it; it does not need to run to make the point.

In [ ]:
# ==========================================================
# 5. Guardrails: the shape of an input check (illustrative)
# ----------------------------------------------------------
# Defines:
#   - stay_on_topic() : Trip the guardrail if the user asks about something off-topic.
# ==========================================================
from agents import input_guardrail, GuardrailFunctionOutput

@input_guardrail                                  # runs BEFORE the agent, on the incoming request
def stay_on_topic(ctx, agent, user_input):
    """Trip the guardrail if the user asks about something off-topic."""
    banned = "medical advice"                     # a toy rule; real ones can call a model to judge
    tripped = banned in user_input.lower()        # True -> the guardrail fires and stops the run
    return GuardrailFunctionOutput(
        output_info={"reason": banned} if tripped else {},
        tripwire_triggered=tripped,               # when True, the SDK halts before the model runs
    )

# Attach the guardrail to an agent via input_guardrails (output_guardrails works the same way)
guarded_agent = Agent(
    name="GuardedAssistant",
    instructions="You are a helpful support assistant. Stay on supported topics.",
    model=DEFAULT_MINI_MODEL,
    input_guardrails=[stay_on_topic],             # checked on every request before the agent answers
)

# A safe request passes the guardrail and runs normally.
safe = await Runner.run(guarded_agent, "How do I reset my password?")
pretty_print(safe.final_output, title="Guardrail passed — request was on-topic")

Now the important half: **watch it fire.** A request that hits the banned topic should be stopped *before* the agent runs. When a tripwire triggers, the SDK raises `InputGuardrailTripwireTriggered`, so we wrap the run in `try/except` and handle the block gracefully instead of crashing.

In [ ]:
# ==========================================================
# 5b. Watch the guardrail FIRE (fail fast, caught cleanly)
# ==========================================================
from agents import InputGuardrailTripwireTriggered   # raised when a tripwire triggers

try:
    # This request contains the banned topic, so the guardrail should stop it.
    await Runner.run(guarded_agent, "Give me medical advice about my symptoms.")
    pp({"status": "run completed", "note": "guardrail did NOT fire"}, title="unexpected")   # should not reach here
except InputGuardrailTripwireTriggered:
    # The SDK halted BEFORE the model ran. No tokens spent on a bad request.
    pp({"tripwire": "FIRED",
        "blocked request": "Give me medical advice about my symptoms.",
        "why": "input guardrail 'stay_on_topic' matched a banned topic"},
       title="🛡️ Guardrail blocked the request before the agent ran")

## 🧱 6. Structured output: typed results with `output_type`

Prose answers are fine for humans, but an *application* needs data it can branch on. Just like JSON mode in M3, an agent can return a **typed object**: define a Pydantic model and pass it as `output_type=...`. The SDK constrains the model to that shape and `result.final_output` comes back as a real instance of your class, not a string. That is what lets an agent's decision drive the next step of a program.

In [ ]:
# ==========================================================
# 6. Structured output: an agent that returns a typed object
# ----------------------------------------------------------
# Defines:
#   - Triage (BaseModel) : the exact shape we want back
#   - classifier         : an agent whose output_type is Triage
# ==========================================================
from pydantic import BaseModel

class Triage(BaseModel):                 # the schema the agent MUST fill
    category: str                        # e.g. "billing" or "technical"
    urgency: str                         # "low" | "medium" | "high"
    reason: str                          # one-line justification

classifier = Agent(
    name="Classifier",
    instructions="Classify the user's support message. Set category, urgency, and a one-line reason.",
    model=DEFAULT_MINI_MODEL,
    output_type=Triage,                  # forces a typed Triage object, not free text
)

result = await Runner.run(classifier, "My payment failed three times and I still got charged!")
ticket = result.final_output             # this is a Triage INSTANCE, not a string
pp(ticket.model_dump(), title="🧱 Structured triage (typed object)")
pp({"code can branch on these fields": {"category": ticket.category, "urgency": ticket.urgency}},
   title="usable in a program")

## 🚀 7. Capstone: a mini customer-support app

Time to assemble the whole vocabulary into one small application. This is the shape of a real agentic system:

- an **input guardrail** rejects off-topic or unsafe requests before any model runs,
- a **triage** agent routes each request to the right **specialist** (handoff),
- each specialist carries a **real tool** it can call to actually resolve the request,
- and we **watch the loop** to see the routing and tool calls the system performed.

It is still one `await Runner.run(...)` call per request. You describe the system; the Runner runs it.

In [ ]:
# ==========================================================
# 7. Capstone: build the support app (tools + specialists + triage + guardrail)
# ----------------------------------------------------------
# Defines:
#   - lookup_order(), issue_refund(), troubleshoot() : the specialists' real tools
#   - support_app : a triage agent that guards input and hands off to specialists
# ==========================================================
# Real tools the specialists can call (faked data so the lab always runs).
@function_tool
def lookup_order(order_id: str) -> str:
    """Look up the status of a customer order by its ID (e.g. 'A100')."""
    orders = {"A100": "shipped, arriving Friday", "B200": "still processing", "C300": "delivered last week"}
    return orders.get(order_id, "no order found with that ID")

@function_tool
def issue_refund(order_id: str) -> str:
    """Check refund eligibility and issue a refund for an order if allowed."""
    eligible = {"A100": True, "B200": True, "C300": False}
    if eligible.get(order_id) is True:
        return f"refund issued for {order_id}"
    if eligible.get(order_id) is False:
        return f"{order_id} is outside the 30-day refund window"
    return "no order found with that ID"

@function_tool
def troubleshoot(problem: str) -> str:
    """Return quick troubleshooting steps for a described technical problem."""
    return f"Steps for '{problem}': 1) update the app, 2) restart, 3) clear cache, 4) contact us if it persists."

# Specialists, each with its own real tool.
billing = Agent(name="Billing", model=DEFAULT_MINI_MODEL,
    instructions="Handle billing. Use lookup_order and issue_refund to resolve charge and refund questions.",
    tools=[lookup_order, issue_refund])

tech = Agent(name="TechSupport", model=DEFAULT_MINI_MODEL,
    instructions="Handle technical issues. Use troubleshoot to give clear steps.",
    tools=[troubleshoot, lookup_order])

# The front door: triage with an input guardrail plus handoffs to the specialists.
support_app = Agent(
    name="SupportFrontDesk",
    model=DEFAULT_MINI_MODEL,
    instructions=("Route the customer to the right specialist. Billing, charges, or refunds go to Billing. "
                  "Bugs, errors, or how-to questions go to TechSupport. Do not answer yourself; hand off."),
    input_guardrails=[stay_on_topic],          # reuse the guardrail from section 5
    handoffs=[billing, tech],
)
pp({"app": "SupportFrontDesk", "guardrail": "stay_on_topic",
    "routes to": ["Billing (lookup_order, issue_refund)", "TechSupport (troubleshoot, lookup_order)"]},
   title="🚀 Support app assembled")

In [ ]:
# ==========================================================
# 7b. Run the support app on real requests (and watch it route)
# ==========================================================
from agents import InputGuardrailTripwireTriggered

requests = [
    "I was charged for order B200 but I want a refund.",
    "The export button throws an error every time. How do I fix it?",
]
for q in requests:
    res = await Runner.run(support_app, q)
    pp({"request": q,
        "handled by": res.last_agent.name,      # which specialist ended up answering
        "answer": res.final_output}, title="🎫 Support app")
    pp(trace_steps(res), title="🔎 loop: routing + tool calls")

# Prove the guardrail still protects the whole app at its front door.
try:
    await Runner.run(support_app, "Give me medical advice about my headache.")
except InputGuardrailTripwireTriggered:
    pp({"off-topic request": "blocked at the front desk"}, title="🛡️ App-level guardrail held")

In [ ]:
# ==========================================================
# 7c. Visualize the app as a graph (who routes to whom)
# ----------------------------------------------------------
# Purpose: draw the support app's architecture so the routing is obvious at a
#          glance. graphviz is preinstalled in Colab and renders inline.
# ==========================================================
from graphviz import Digraph

g = Digraph(graph_attr={"rankdir": "LR", "bgcolor": "transparent"})
g.attr("node", style="filled", fontname="Helvetica", fontsize="11")

g.node("user",    "User request",                              shape="oval", fillcolor="#eef2ff")
g.node("guard",   "Input guardrail\n(stay_on_topic)",          shape="box",  fillcolor="#fde8e8")
g.node("triage",  "SupportFrontDesk\n(triage)",                shape="box",  fillcolor="#dbeafe")
g.node("billing", "Billing\n[lookup_order, issue_refund]",     shape="box",  fillcolor="#dcfce7")
g.node("tech",    "TechSupport\n[troubleshoot, lookup_order]", shape="box",  fillcolor="#dcfce7")

g.edge("user", "guard")
g.edge("guard", "triage", label="passes")
g.edge("triage", "billing", label="billing / refunds")
g.edge("triage", "tech",    label="bugs / how-to")

g   # Colab renders the graph inline


## 🔭 Bonus: observability with tracing

Every run you executed was **automatically traced** by the SDK. Open [platform.openai.com/traces](https://platform.openai.com/traces) and you will see each run laid out as a timeline: the agent, its tool calls, handoffs, and guardrail checks. For a multi-step workflow you can group several runs under one named trace:

```python
from agents import trace

with trace("customer support session"):
    await Runner.run(support_app, "I was charged twice.")
    await Runner.run(support_app, "The export button errors.")
```

That grouping is how you debug and monitor an agent in production, the same way you read logs for any other service.

## 🛠️ 8. Hands-on: extend the app (build your own tool)

Put it together. Write **one** new tool with `@function_tool`, attach it to an agent, and ask a question that should trigger it. Ideas: `convert_currency(amount, from_ccy, to_ccy)`, `word_count(text)`, or `days_until(date)`. Remember the two rules: **type hints** for the arguments, and a **clear docstring** so the model knows when to call it.

Bonus: add a third specialist (say a **Shipping** agent with a `track_package` tool) to `support_app`, then re-run section 7b and watch triage route a shipping question to it.

In [ ]:
# ==========================================================
# 8. Hands-on: your own @function_tool + agent (worked example)
# ----------------------------------------------------------
# Defines:
#   - c_to_f() : Convert a temperature in Celsius to Fahrenheit.
# ==========================================================
# A complete example: a tool that converts Celsius to Fahrenheit.
@function_tool
def c_to_f(celsius: float) -> str:            # typed argument the model must supply
    """Convert a temperature in Celsius to Fahrenheit."""
    return f"{celsius}C is {celsius * 9 / 5 + 32}F"   # a value the agent can use in its answer

# Build an agent that is allowed to use your tool
my_agent = Agent(
    name="Converter",                         # any name you like
    instructions="You convert temperatures. Use the c_to_f tool whenever asked to convert.",
    model=DEFAULT_MINI_MODEL,
    tools=[c_to_f],                           # pass your tool function here
)

# Ask a question that should make the agent call your tool
result = await Runner.run(my_agent, "What is 25 degrees Celsius in Fahrenheit?")
pretty_print(result.final_output, title="Your agent in action")

# YOUR TURN: replace c_to_f with your own tool (e.g. word_count or days_between),
# following the same pattern, then ask a question that needs it.

## 🗺️ Where this fits: Agents SDK vs LangChain / LangGraph / CrewAI

You now know four ways to build LLM systems in this course. They are less competitors than different altitudes. Rule of thumb: reach for the OpenAI Agents SDK when you want the least code to ship an agent, and reach for LangGraph when you need fine grained control over a stateful, looping workflow.

| Framework | What it is | Best for | State and control | Provider |
|---|---|---|---|---|
| **OpenAI Agents SDK** (this lab) | Minimal agent runtime: Agent, Runner, Tools, Handoffs, Guardrails | Standing up single or multi agent apps fast, little code | Runner drives the loop; handoffs route; less explicit control | OpenAI centric (other models via LiteLLM) |
| **LangChain** (M04) | Broad toolkit: chains / LCEL, integrations, memory, retrievers | Pipelines and RAG, huge integration ecosystem | Mostly linear chains; not built for cyclic agent control | Provider agnostic |
| **LangGraph** (M06) | Graph orchestration: nodes and edges over explicit shared state | Complex, controllable flows: loops, branching, human in the loop, durability | The most control (explicit graph plus state) | Agnostic (built on LangChain) |
| **CrewAI** (M08) | Role based agent crews: roles, tasks, process | Collaborative role playing agent teams | A crew process orchestrates the roles | Agnostic |

**Takeaway.** Same job, different trade off: the Agents SDK minimizes boilerplate, LangGraph maximizes control, LangChain maximizes integrations, and CrewAI leans into role based teams. Pick by how much control the problem actually needs.

## 🎯 Wrap-up

You built agents with the OpenAI Agents SDK using its whole vocabulary:

- **Agent** — a model plus a name and instructions, defined in a couple of lines.
- **Runner** — `await Runner.run(agent, "...")` drives the agentic loop and returns `result.final_output` (use `Runner.run_sync(...)` in scripts, `await Runner.run(...)` in notebooks).
- **Tools** — a plain Python function plus `@function_tool`; the SDK builds the schema from your type hints and docstring, no hand-written JSON.
- **Handoffs** — a triage agent lists specialists in `handoffs=[...]` and routes each request at runtime, all inside one run.
- **Guardrails** — `input_guardrails` / `output_guardrails` fail fast on unsafe or off-topic input and output (you watched one actually fire).
- **Structured output** with `output_type` returns a typed Pydantic object your program can branch on, not just prose.
- **A real app** — you assembled triage + specialists + tools + a firing guardrail into one support assistant, all run by a single `await Runner.run(...)`.

Compare this to the raw function-calling loop you wrote by hand in Module 3: same mechanics, a fraction of the code. That is the trade the SDK makes — you describe *what* your system is, and the Runner handles *how* it executes. Next module you tie this together with MCP (agent-to-tools) and A2A (agent-to-agent) into a full production system.